# Quick Start Guide v3

This notebook runs the current vulnerability analysis workflow from the repository README: the NeMo Agent Toolkit `cve_pipeline` graph. The v3 flow analyzes one or more CVE or GHSA identifiers against one container image, then writes per-agent Markdown reports and final VEX JSON classifications.

The pipeline consumes a validated `PipelineInput` JSON document containing `vulns`, `image_name`, and `image_tag`; it no longer needs an SBOM. If `filesystem_path` is omitted, the orchestrator pulls and extracts a flattened rootfs via `crane` and reuses the image cache on later runs. You can still provide a pre-extracted `filesystem_path` when you want to manage extraction yourself.

For deeper usage details, see the [README](../README.md).

**Cell classifications:** every code cell is classified in notebook metadata as `setup-only`, `executable`, or `illustrative`. Setup-only cells initialize state, executable cells are expected to run as written in a configured environment (some are explicitly opt-in), and illustrative cells only prepare or show examples without launching work.


## Prerequisites

Before running the workflow cells, make sure the project is installed and configured:

- Python 3.13 and project dependencies installed with `uv sync --locked --group dev` (this includes `IPython`)
- A C++ compiler available during installation: Xcode Command Line Tools on macOS or `build-essential` on Ubuntu/Debian
- `NVIDIA_API_KEY` configured for the NAT `nim` LLM
- `TAVILY_API_KEY` configured for web search
- `crane` installed when `filesystem_path` is omitted
- Optional registry auth in `~/.docker/config.json`, using `crane auth login <registry>` when needed

Both credentials are required for a complete, uncached run. They may be exported, stored in a project-root `.env`, or injected through the notebook environment, an orchestrator, or a secret manager; never commit real credentials. Compatible proxies may be selected with `NV_BASE_URL` and `TAVILY_API_BASE_URL`, but the key variables must still contain the credentials those endpoints expect.

This notebook is standalone and stateful. Run it from top to bottom. After a kernel restart, rerun repository setup, configuration, and all producer cells rather than relying on files from an earlier run. The default input is the checked-in, single-vulnerability Morpheus quick-start fixture; edit the input cell when you want to analyze a different image or identifier.


In [ ]:
from __future__ import annotations

import json
import os
import shlex
import shutil
import subprocess
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from IPython.display import Markdown, display

from vuln_analysis.data_models.input import CVEEntry, PipelineInput
from vuln_analysis.data_models.output import PipelineResult
from vuln_analysis.preflight import run_live_preflight
from vuln_analysis.semantic_failure import require_successful_pipeline_result


## Set Repository Root

Change the notebook working directory to the repository root. All relative paths below are resolved from `REPO_ROOT`.


In [ ]:
def find_repo_root(start=None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / 'pyproject.toml').exists() and (candidate / 'README.md').exists():
            return candidate
    raise RuntimeError('Could not find the vulnerability-analysis repository root.')


REPO_ROOT = find_repo_root()
os.chdir(REPO_ROOT)
load_dotenv(REPO_ROOT / '.env')
print(f'REPO_ROOT = {REPO_ROOT}')


## Check Local Setup

This cell only checks local files, commands, and environment variables. It does not pull images or call any LLMs.


In [ ]:
def resolve_vuln_analysis_command() -> list:
    if shutil.which('vuln-analysis'):
        return ['vuln-analysis']
    if shutil.which('uv'):
        return ['uv', 'run', 'vuln-analysis']
    raise RuntimeError('Could not find `vuln-analysis` or `uv` on PATH.')


VULN_ANALYSIS_CMD = resolve_vuln_analysis_command()
print('vuln-analysis command:', shlex.join(VULN_ANALYSIS_CMD))

setup_rows = [
    {'check': 'c++', 'ok': bool(shutil.which('c++')), 'required': True},
    {'check': 'configs/config.yml', 'ok': Path('configs/config.yml').exists(), 'required': True},
    {'check': 'data/examples/quick_start_input.json', 'ok': Path('data/examples/quick_start_input.json').exists(), 'required': True},
    {'check': 'NVIDIA_API_KEY', 'ok': bool(os.getenv('NVIDIA_API_KEY')), 'required': True},
    {'check': 'TAVILY_API_KEY', 'ok': bool(os.getenv('TAVILY_API_KEY')), 'required': True},
    {'check': 'crane', 'ok': bool(shutil.which('crane')), 'required': 'automatic extraction only'},
]
setup_df = pd.DataFrame(setup_rows)
display(setup_df)

missing_required = [row['check'] for row in setup_rows if row['required'] is True and not row['ok']]
if missing_required:
    print('Set or install these before running the full pipeline:', ', '.join(missing_required))


# Choose Container Input Mode

`PipelineInput` validates these required fields:

- `vulns`: a list of vulnerability entries, each with `vuln_id`
- `image_name`: the image repository/name without the tag
- `image_tag`: the image tag

- Leave `filesystem_path` unset for automatic extraction. The orchestrator pulls `image_name:image_tag` with `crane`, extracts the rootfs into `${VULN_ANALYSIS_IMAGE_CACHE:-.cache/vuln_analysis/images}/<digest>/rootfs`, and records the resolved paths internally.
- Set `filesystem_path` when you already have a merged rootfs. `image_config_path` is optional in this mode.

The default cell loads the same known-good, single-vulnerability Morpheus fixture used by the README and uses automatic extraction. Replace its values when you want to analyze a different target.


In [ ]:
fixture_path = REPO_ROOT / 'data' / 'examples' / 'quick_start_input.json'
fixture_input = PipelineInput.model_validate_json(fixture_path.read_text(encoding='utf-8'))
image_name = fixture_input.image_name
image_tag = fixture_input.image_tag
vulnerabilities = fixture_input.vulns

# Set these only when reusing a pre-extracted image.
filesystem_path: Path | None = None
image_config_path: Path | None = None

quick_start_dir = REPO_ROOT / '.tmp' / 'quick_start_v3'
output_dir = quick_start_dir / 'outputs'
standalone_input_dir = quick_start_dir / 'standalone_inputs'
input_file = quick_start_dir / 'pipeline_input.json'
result_file = quick_start_dir / 'pipeline_result.json'
run_log_path = quick_start_dir / 'nat_run.log'


## Optional: Pre-extract an Image

You can skip this section when `filesystem_path` is unset. To warm the cache or debug image extraction separately, run the standalone helper from a terminal:

```bash
uv run python scripts/extract_image.py <container-image>:<tag> /tmp/img-cache
```

The next cell prints the matching command. If you run the helper manually, set `filesystem_path` and `image_config_path` above to the `rootfs_path` and `config_path` printed by the helper.


In [ ]:
image_ref = f'{image_name}:{image_tag}'
extract_cmd = [
    'uv', 'run', 'python', 'scripts/extract_image.py',
    image_ref,
    str(REPO_ROOT / '.cache' / 'vuln_analysis' / 'images'),
]
print('$ ' + shlex.join(extract_cmd))
print('Run this manually only if you want to pre-extract the configured image.')


# Build `PipelineInput`

A `PipelineInput` represents one or more CVE/GHSA identifiers against one container image. Pydantic normalizes and validates every identifier immediately. This cell first rejects any untouched tutorial placeholders so they cannot be mistaken for runnable defaults.


In [ ]:
pipeline_input = PipelineInput(
    vulns=vulnerabilities,
    image_name=image_name,
    image_tag=image_tag,
    filesystem_path=str(filesystem_path) if filesystem_path else '',
    image_config_path=str(image_config_path) if image_config_path else None,
    output_dir=str(output_dir),
    cache=True,
    skip_web_research=False,
)

quick_start_dir.mkdir(parents=True, exist_ok=True)
input_file.write_text(
    pipeline_input.model_dump_json(indent=2, exclude_none=True) + '\n',
    encoding='utf-8',
)

print(f'Wrote {input_file.relative_to(REPO_ROOT)}')
print(input_file.read_text(encoding='utf-8'))


## Validate Input and Prerequisites

This runs the same explicit live preflight exposed by `vuln-analysis preflight --input-file <file> --json` and `POST /preflight`. It validates the input, credentials, provider URL syntax, local paths, and `crane`, then makes one credential-only readiness request to each provider. It performs no inference or search work.


In [ ]:
preflight = await run_live_preflight(pipeline_input)
display(pd.DataFrame([check.model_dump() for check in preflight.checks]))
if not preflight.ready:
    raise RuntimeError(preflight.failure_message())

print('PipelineInput and prerequisites are ready.')


# Run the Full Pipeline

This invokes `vuln-analysis run-pipeline`, which runs the registered NAT `cve_pipeline` workflow and saves `pipeline_input.json`, `pipeline_result.json`, and `nat_run.log` together. First runs can take several minutes because they may pull the image, perform web research, and call LLMs. Later runs can reuse the image cache and per-stage report cache.


In [ ]:
config_file = Path('configs/config.yml')
cmd = VULN_ANALYSIS_CMD + [
    'run-pipeline',
    '--pipeline-input', str(input_file),
    '--config-file', str(config_file),
    '--artifacts-dir', str(quick_start_dir),
]

print('$ ' + shlex.join(str(part) for part in cmd))
completed = subprocess.run(
    [str(part) for part in cmd],
    cwd=REPO_ROOT,
    text=True,
)
completed.check_returncode()
print(f'Run log: {run_log_path.relative_to(REPO_ROOT)}')


## Load the Saved Pipeline Result

`run-pipeline` validates and saves the last complete `PipelineResult` emitted by NAT. This cell reloads that artifact and requires top-level/per-vulnerability success, all required stages and report paths, VEX fields, and VEX JSON paths.


In [ ]:
pipeline_result = require_successful_pipeline_result(
    PipelineResult.model_validate_json(result_file.read_text(encoding='utf-8'))
)
print(f'Loaded {result_file.relative_to(REPO_ROOT)}')
print(result_file.read_text(encoding='utf-8'))


## Pipeline Summary


In [ ]:
first_result = pipeline_result.results[0]
summary = {
    'image_name': pipeline_result.image_name,
    'image_tag': pipeline_result.image_tag,
    'success': pipeline_result.success,
    'elapsed_seconds': round(pipeline_result.elapsed_seconds, 1),
    'vulnerability_count': len(pipeline_result.results),
}
display(pd.DataFrame([summary]))

vuln_rows = [
    {
        'vuln_id': result.vuln_id,
        'success': result.success,
        'vex_status': result.vex_status,
        'vex_justification': result.vex_justification,
        'error': result.error,
    }
    for result in pipeline_result.results
]
display(pd.DataFrame(vuln_rows))

stage_rows = []
if pipeline_result.container_analyzer:
    stage_rows.append({
        'vuln_id': None,
        'stage': 'container_analyzer',
        **pipeline_result.container_analyzer.model_dump(),
    })

for result in pipeline_result.results:
    for name in ['cve_researcher', 'exploitability_analyzer', 'vex_categorizer']:
        stage = getattr(result, name)
        if stage:
            stage_rows.append({'vuln_id': result.vuln_id, 'stage': name, **stage.model_dump()})

display(pd.DataFrame(stage_rows))


# Inspect Outputs

The orchestrator writes outputs under the `output_dir` from `PipelineInput`:

- `vulnerability_reports/`
- `container_reports/`
- `exploitability_reports/`
- `vex_results/`


In [ ]:
resolved_output_dir = Path(pipeline_input.output_dir)
if not resolved_output_dir.is_absolute():
    resolved_output_dir = REPO_ROOT / resolved_output_dir

if not resolved_output_dir.exists():
    raise FileNotFoundError(f'Output directory not found: {resolved_output_dir}')

output_files = sorted(path for path in resolved_output_dir.rglob('*') if path.is_file())
for path in output_files:
    print(path.relative_to(REPO_ROOT))


## Load Final VEX JSON


In [ ]:
vex_json_path = None
vex_stage = first_result.vex_categorizer
json_path = vex_stage.extra.get('json_path') if vex_stage else None
if json_path:
    candidate = Path(json_path)
    vex_json_path = candidate if candidate.is_absolute() else REPO_ROOT / candidate

if not vex_json_path or not vex_json_path.exists():
    vex_json_files = sorted(
        (resolved_output_dir / 'vex_results').glob('*.json'),
        key=lambda path: path.stat().st_mtime,
        reverse=True,
    )
    if not vex_json_files:
        raise FileNotFoundError(f"No VEX JSON files found under {resolved_output_dir / 'vex_results'}")
    vex_json_path = vex_json_files[0]

vex_result = json.loads(vex_json_path.read_text(encoding='utf-8'))
print(f'Loaded {vex_json_path.relative_to(REPO_ROOT)}')

display(pd.DataFrame([{
    'vuln_id': vex_result.get('vuln_id') or vex_result.get('cve_id') or vex_result.get('ghsa_id'),
    'container': vex_result.get('container'),
    'status': vex_result.get('status'),
    'justification': vex_result.get('justification'),
    'exploitability_verdict': vex_result.get('exploitability_verdict'),
    'confidence': vex_result.get('confidence'),
    'risk_level': vex_result.get('risk_level'),
    'container_type': vex_result.get('container_type'),
}]))

print(json.dumps(vex_result, indent=2))


## View VEX Markdown Report


In [ ]:
if not vex_stage or not vex_stage.report_path:
    raise FileNotFoundError('The VEX stage did not produce a Markdown report.')

report_path = Path(vex_stage.report_path)
if not report_path.is_absolute():
    report_path = REPO_ROOT / report_path

print(report_path)
display(Markdown(report_path.read_text(encoding='utf-8')))


# Illustrative: Run More Vulnerabilities

One `PipelineInput` can analyze several identifiers against the same image. The illustrative cell below prepares the three reviewed Morpheus eval cases without launching them.


In [ ]:
batch_input = pipeline_input.model_copy(update={
    'vulns': [
        CVEEntry(vuln_id='GHSA-53q9-r3pm-6pq6'),
        CVEEntry(vuln_id='GHSA-ph84-rcj2-fxxm'),
        CVEEntry(vuln_id='CVE-2024-9287'),
    ],
})

display(pd.DataFrame([entry.model_dump() for entry in batch_input.vulns]))
print(batch_input.model_dump_json(indent=2, exclude_none=True))


# Optional: Individual Agents

The checked-in JSON files remain useful as schema examples, but their `_comment` fields mark placeholder paths as non-runnable. The next cell executes standalone-input generation from the artifacts saved above. It derives the selected identifier from the actual `PipelineResult`, so it cannot drift from the vulnerability that was run.

The generator writes every input supported by available prior-stage reports, including a reproduction input when a later stage fails. Run the generated files with the corresponding standalone NAT configurations documented in the README.


In [ ]:
standalone_vuln_id = pipeline_result.results[0].vuln_id
generate_cmd = VULN_ANALYSIS_CMD + [
    'generate-standalone-inputs',
    '--pipeline-input', str(input_file),
    '--pipeline-result', str(result_file),
    '--vuln-id', standalone_vuln_id,
    '--output-dir', str(standalone_input_dir),
]

print(f'Selected vulnerability: {standalone_vuln_id}')
print('$ ' + shlex.join(str(part) for part in generate_cmd))
generated = subprocess.run([str(part) for part in generate_cmd], cwd=REPO_ROOT, text=True)
generated.check_returncode()

generated_files = sorted(path for path in standalone_input_dir.glob('*.json') if path.is_file())
if not generated_files:
    raise FileNotFoundError(f'No standalone inputs were written to {standalone_input_dir}')
display(pd.DataFrame({'standalone_input': [str(path.relative_to(REPO_ROOT)) for path in generated_files]}))


# Illustrative: Optional Manual HTTP Server

The primary client-server path is the Docker Compose API in the README. For optional host-side debugging after local setup, start the NAT server from a separate terminal:

```bash
nat serve --config_file=configs/config.yml --host 0.0.0.0 --port 26466
```

Send the same `PipelineInput` JSON body to `POST /preflight` first. HTTP 200 means ready; HTTP 503 includes per-check remediation. Then submit it to the synchronous or asynchronous workflow endpoint.


In [ ]:
# Illustrative only. The routes below are supported by the shipped NAT 1.8 configuration.
# import requests
# payload = pipeline_input.model_dump(mode='json', exclude_none=True)
# preflight_response = requests.post('http://localhost:26466/preflight', json=payload, timeout=30)
# preflight_response.raise_for_status()
# response = requests.post('http://localhost:26466/generate', json=payload, timeout=3600)
# response_payload = response.json()
# successful_result = require_successful_pipeline_result(response_payload)
# response.raise_for_status()
# successful_result


# Cleanup Notes

This notebook writes quick-start inputs, results, logs, generated standalone files, and reports under `.tmp/quick_start_v3/`. Automatic image extraction caches live under `${VULN_ANALYSIS_IMAGE_CACHE:-.cache/vuln_analysis/images}`. Per-agent outputs are regular Markdown and JSON files, so you can inspect, archive, diff, or delete them directly. Use `uv run vuln-analysis cache info` and the README's dry-run-first `cache prune` workflow for application caches. Set `cache` to `False` in `PipelineInput` when you want fresh agent reports.
